In [1]:
# %pip install python-dotenv
# %uv add dspy

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))


In [3]:
import time
import threading
import tiktoken
from collections import deque
import dspy
from dspy.utils.callback import BaseCallback


class SlidingWindowLimiter:
    """Rate limiter that enforces both request and token limits per rolling minute."""

    _instance = None
    _lock = threading.Lock()

    def __new__(cls, max_requests_per_min=1000, max_tokens_per_min=2_000_000):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.max_requests = max_requests_per_min
            cls._instance.max_tokens = max_tokens_per_min
            cls._instance.requests = deque()  # [(timestamp, tokens)]
            cls._instance._lock = threading.Lock()
            cls._instance.encoder = tiktoken.get_encoding("cl100k_base")
        return cls._instance

    def _cleanup(self, now):
        """Remove entries older than 60s."""
        while self.requests and now - self.requests[0][0] > 60:
            self.requests.popleft()

    def _count(self):
        """Total requests & tokens in current 60s window."""
        total_tokens = sum(t for _, t in self.requests)
        return len(self.requests), total_tokens

    def acquire(self, tokens_used=0):
        """Wait until request fits in sliding 60s window."""
        with self._lock:
            while True:
                now = time.time()
                self._cleanup(now)
                req_count, token_count = self._count()

                # Can fit in current 60s window?
                if (req_count < self.max_requests and
                        token_count + tokens_used <= self.max_tokens):
                    # Record the new request
                    self.requests.append((now, tokens_used))
                    break  # proceed

                # Otherwise, figure out when we can retry
                oldest_time = self.requests[0][0]
                sleep_time = max(0.01, 60 - (now - oldest_time))
                print(f"⚠️ Throttling: sleeping {sleep_time:.2f}s (req={req_count}, tokens={token_count})")
                time.sleep(sleep_time)


class DelayAndLogCallback(BaseCallback):
    """DSPy callback using sliding window limiter."""

    def __init__(self):
        self.limiter = SlidingWindowLimiter()

    def _estimate_tokens(self, messages=None, prompt=None):
        """Estimate token usage using tiktoken."""
        text = ""
        if prompt:
            text = str(prompt)
        elif messages:
            # concatenate all message contents
            text = " ".join(m.get("content", "") for m in messages)
        return len(self.limiter.encoder.encode(text))

    def on_lm_start(self, *args, **kwargs):
        inputs = kwargs.get("inputs") or {}
        prompt = inputs.get("prompt")
        messages = inputs.get("messages")
        tokens_used = self._estimate_tokens(messages=messages, prompt=prompt)
        self.limiter.acquire(tokens_used=tokens_used)

    def on_lm_end(self, *args, **kwargs):
        pass


In [4]:
import os
# os.environ['OPENAI_API_KEY'/] = input()
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [5]:
import dspy
lm = dspy.LM("azure/gpt-4.1", temperature=1.0, num_retries=30, callbacks=[DelayAndLogCallback()])
tlm = dspy.LM("azure/gpt-4.1",temperature=1.0)
dspy.configure(lm=lm)

In [6]:
# print(lm("Say this is a test!") ) # => ['This is a test!']
print(lm(messages=[{"role": "user", "content": "Say this is a test!"}]))  # => ['This is a test!']
print(tlm(messages=[{"role": "user", "content": "t : Say this is a test!"}]))  # => ['This is a test!']

['This is a test!']
['This is a test!']


## Load the benchmark and view one example from the benchmark

In [7]:
from gepa_artifact.benchmarks.researchcodebench import benchmark as rc_metas

In [8]:
bench = rc_metas[0].benchmark()

In [9]:
len(bench.train_set), len(bench.val_set), len(bench.test_set)

(34, 34, 144)

In [10]:
import pprint
pprint.pprint(bench.train_set[0])

Example({'paper_id': 'DiffusionDPO', 'problem_root_rel': 'pset/DiffusionDPO', 'annotated_file_path': 'loss.py', 'snippet_name': 'calculate model losses', 'start_line': 33, 'end_line': 39, 'masked_file': 'import torch\nimport torch.nn.functional as F\n\ndef compute_loss(model_pred, target, args, ref_unet=None, model_batch_args=None, added_cond_kwargs=None):\n    """\n    Compute the loss for either SFT or DPO training.\n    \n    Args:\n        model_pred: The prediction from the model being trained\n        target: The target (typically noise) the model is trying to predict\n        args: The arguments containing training configuration\n        ref_unet: The reference UNet model (required for DPO)\n        model_batch_args: Arguments to pass to the UNet models (required for DPO)\n        added_cond_kwargs: Additional conditioning kwargs (required for DPO with SDXL)\n    \n    Returns:\n        loss: The computed loss\n        metrics: Dictionary of additional metrics for logging\n     

## Load the program and display the program
The program is a 3-module system, each of which handles the urgency, sentiment and categories classification respectively

In [11]:
program = rc_metas[0].program[0]
program

model = Predict(RCBResponse(instance_id, paper_id, snippet_name, masked_file, context_files, paper -> result
    instructions='Solve the question and provide the answer in the correct format.'
    instance_id = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Instance Id:', 'desc': '${instance_id}'})
    paper_id = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Paper Id:', 'desc': '${paper_id}'})
    snippet_name = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Snippet Name:', 'desc': '${snippet_name}'})
    masked_file = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Masked File:', 'desc': '${masked_file}'})
    context_files = Field(annotation=list required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Context Files:', 'desc': '${context_files}'})
    paper = Field(annotation=st

### Make Sure docker is installed and running

## Define an evaluator and evaluate the base program

In [ ]:
import dspy
evaluate = dspy.Evaluate(
    devset=bench.test_set,
    metric=rc_metas[0].metric,
    num_threads=4,
    display_table=True,
    display_progress=True,
    max_errors=100 * len(bench.test_set),
    provide_traceback = True,
    failure_score=0,
    save_as_json='rcb.json',
    save_as_csv='rcb.csv'
)

## Load the GEPA Optimizer

In [12]:

import dspy
from gepa_artifact.gepa.gepa import GEPA,GEPAState
from gepa_artifact.utils.capture_stream_logger import Logger

import time

In [13]:


runs_dir = os.path.join(os.getcwd(), "runs", time.strftime("%Y-%m-%d_%H-%M-%S"))
os.makedirs(runs_dir, exist_ok=True)

gepa_logger = Logger(os.path.join(runs_dir, "run_log.txt"))

if rc_metas[0].feedback_fn_maps is None or rc_metas[0].feedback_fn_maps[0] is None:
    def feedback_func(predictor_output, predictor_inputs, module_inputs, module_outputs, captured_trace):
        pred = rc_metas[0].metric_with_feedback(module_inputs, module_outputs, None)
        return {
            "feedback_score": pred.score,
            "feedback_text": pred.feedback,
        }

    feedback_fn_map = {k:feedback_func for k, v in program.named_predictors()}
else:
    feedback_fn_map = rc_metas[0].feedback_fn_maps[0]

optimizer = GEPA(
    named_predictor_to_feedback_fn_map=feedback_fn_map,
    knowledgebase_qe=None,
    metric=rc_metas[0].metric,
    run_linearized_gepa=False,
    use_merge=True, 
    teacher_lm = tlm,
    set_for_merge_minibatch='val', 
    track_scores_on='val',
    num_iters=20,
    run_dir=runs_dir,
    logger=gepa_logger,
    num_threads=4)

## Optimize the program with GEPA

In [14]:
rc_metas[0].program[0].get_lm()

## Load from the Saved dir

In [15]:
state = GEPAState.load('runs/rcb-w-gold')

In [16]:
state.total_num_evals

608

In [17]:
def idxmax(lst):
    """Return the index of the maximum value in a list."""
    max_val = max(lst)
    return lst.index(max_val)

In [18]:
gepa_state = state
best_prog_idx = idxmax(gepa_state.per_program_tracked_scores)
best_progs = gepa_state.program_candidates
best_prog = best_progs[best_prog_idx]

In [19]:
len(best_progs)

12

In [20]:
# # best_progs
# for idx,i in enumerate(best_progs):
#     print(f'*************{idx}***************')
#     for name, pred in i.named_predictors():
#         print("================================")
#         print(f"Predictor: {name}")
#         print("================================")
#         print("Prompt:")
#         print(pred.signature.instructions)
#         print("*********************************")

In [21]:
optimized_program=best_prog

In [22]:
best_prog_idx

10

### Let's print the prompts that GEPA discovered

In [23]:
for name, pred in optimized_program.named_predictors():
    print("================================")
    print(f"Predictor: {name}")
    print("================================")
    print("Prompt:")
    print(pred.signature.instructions)
    print("*********************************")

Predictor: model
Prompt:
You are to write *only the Python code block(s)* that implement a specifically named masked code block, marked by a "TODO" in a larger Python file. This file is always part of a research codebase for machine learning or deep learning, usually for generative models, probabilistic modeling, or autoregressive transformers, with a style conforming to PyTorch and modern ML conventions. The context for the TODO includes the file's immediate code (class, method, etc), the research paper in LaTeX (with equations, notation, and implementation details), and possibly other key code files for help (such as numpy equivalents, different sampling/solver methods, or loss function utilities).

Your response must strictly replace just the specific TODO block, inserting only the Python code (properly indented, with no extra explanation or docstrings). Do not provide the entire file, nor extra imports, nor header comments: only the block required, with indentation matching functio

## Now, let's evaluate the optimized program

In [24]:
import dspy
evaluate = dspy.Evaluate(
    devset=bench.test_set,
    metric=rc_metas[0].metric,
    num_threads=4,
    display_table=True,
    display_progress=True,
    max_errors=100 * len(bench.test_set),
    provide_traceback = True,
    failure_score=0,
    save_as_json='optimized_rcb.json',
    save_as_csv='optimized_rcb.csv'
)

In [25]:
# latest_program = best_progs[-1]

# for name, pred in latest_program.named_predictors():
#     print("================================")
#     print(f"Predictor: {name}")
#     print("================================")
#     print("Prompt:")
#     print(pred.signature.instructions)
#     print("*********************************")

## if we use the best program given by gepa

In [ ]:
evaluate(optimized_program)

Average Metric: 2.00 / 25 (8.0%):  17%|█▋        | 25/144 [01:01<11:38,  5.87s/it]⚠️ Throttling: sleeping 0.03s (req=45, tokens=1962258)
⚠️ Throttling: sleeping 0.15s (req=45, tokens=1963414)
⚠️ Throttling: sleeping 0.02s (req=45, tokens=1981694)
⚠️ Throttling: sleeping 0.04s (req=45, tokens=1999971)
⚠️ Throttling: sleeping 0.02s (req=44, tokens=1968921)
Average Metric: 2.00 / 27 (7.4%):  18%|█▊        | 26/144 [01:02<09:12,  4.68s/it]⚠️ Throttling: sleeping 0.02s (req=43, tokens=1956185)
⚠️ Throttling: sleeping 3.55s (req=43, tokens=1974456)
Average Metric: 2.00 / 28 (7.1%):  19%|█▉        | 28/144 [01:02<05:29,  2.84s/it]⚠️ Throttling: sleeping 0.01s (req=43, tokens=1992716)
⚠️ Throttling: sleeping 0.05s (req=42, tokens=1961623)
⚠️ Throttling: sleeping 0.03s (req=42, tokens=1979965)
⚠️ Throttling: sleeping 0.08s (req=42, tokens=1998284)
⚠️ Throttling: sleeping 0.02s (req=41, tokens=1967199)
⚠️ Throttling: sleeping 3.08s (req=41, tokens=1985467)
Average Metric: 2.00 / 29 (6.9%):  20%|

2025/11/13 08:49:33 ERROR dspy.utils.parallelizer: Error for Example({'paper_id': 'LEN', 'problem_root_rel': 'pset/LEN', 'annotated_file_path': 'Synthetic.py', 'snippet_name': 'main iteration loop', 'start_line': 122, 'end_line': 164, 'masked_file': 'import argparse\nimport numpy as np \nimport matplotlib.pyplot as plt \nimport random \nimport scipy \nfrom scipy.optimize import fsolve\nimport time \n\n# Keep parser outside if __name__ so it can be imported\nparser = argparse.ArgumentParser(description=\'Minimax Optimization.\')\nparser.add_argument(\'--training_time\',  type=float, default=10.0, help=\'total training time\')\nparser.add_argument(\'--n\', type=int, default=100, help=\'size of problem\')\nparser.add_argument(\'--m\', type=int, default=100, help=\'frequency of lazy Hessian update\')\nparser.add_argument(\'--seed\',  type=int, default=42, help=\'seed of random number\')\nparser.add_argument(\'--eta\', type=float, default=1e-2, help=\'stepsize of extra gradient\')\nparser.a

Average Metric: 23.00 / 143 (16.1%): 100%|██████████| 144/144 [09:06<00:00, 57.99s/it]

2025/11/13 08:51:22 ERROR dspy.utils.parallelizer: Error for Example({'paper_id': 'LEN', 'problem_root_rel': 'pset/LEN', 'annotated_file_path': 'Synthetic.py', 'snippet_name': 'main iteration loop', 'start_line': 122, 'end_line': 164, 'masked_file': 'import argparse\nimport numpy as np \nimport matplotlib.pyplot as plt \nimport random \nimport scipy \nfrom scipy.optimize import fsolve\nimport time \n\n# Keep parser outside if __name__ so it can be imported\nparser = argparse.ArgumentParser(description=\'Minimax Optimization.\')\nparser.add_argument(\'--training_time\',  type=float, default=10.0, help=\'total training time\')\nparser.add_argument(\'--n\', type=int, default=100, help=\'size of problem\')\nparser.add_argument(\'--m\', type=int, default=100, help=\'frequency of lazy Hessian update\')\nparser.add_argument(\'--seed\',  type=int, default=42, help=\'seed of random number\')\nparser.add_argument(\'--eta\', type=float, default=1e-2, help=\'stepsize of extra gradient\')\nparser.a

Average Metric: 23.00 / 143 (16.1%): : 145it [10:55, 73.24s/it]                       

...

Average Metric: 23.00 / 143 (16.1%): : 152it [24:59, 116.70s/it]

2025/11/13 09:07:26 ERROR dspy.utils.parallelizer: Error for Example({'paper_id': 'LEN', 'problem_root_rel': 'pset/LEN', 'annotated_file_path': 'Synthetic.py', 'snippet_name': 'main iteration loop', 'start_line': 122, 'end_line': 164, 'masked_file': 'import argparse\nimport numpy as np \nimport matplotlib.pyplot as plt \nimport random \nimport scipy \nfrom scipy.optimize import fsolve\nimport time \n\n# Keep parser outside if __name__ so it can be imported\nparser = argparse.ArgumentParser(description=\'Minimax Optimization.\')\nparser.add_argument(\'--training_time\',  type=float, default=10.0, help=\'total training time\')\nparser.add_argument(\'--n\', type=int, default=100, help=\'size of problem\')\nparser.add_argument(\'--m\', type=int, default=100, help=\'frequency of lazy Hessian update\')\nparser.add_argument(\'--seed\',  type=int, default=42, help=\'seed of random number\')\nparser.add_argument(\'--eta\', type=float, default=1e-2, help=\'stepsize of extra gradient\')\nparser.a

Average Metric: 23.00 / 143 (16.1%): : 153it [26:59, 117.77s/it]

2025/11/13 09:09:27 ERROR dspy.utils.parallelizer: Error for Example({'paper_id': 'LEN', 'problem_root_rel': 'pset/LEN', 'annotated_file_path': 'Synthetic.py', 'snippet_name': 'main iteration loop', 'start_line': 122, 'end_line': 164, 'masked_file': 'import argparse\nimport numpy as np \nimport matplotlib.pyplot as plt \nimport random \nimport scipy \nfrom scipy.optimize import fsolve\nimport time \n\n# Keep parser outside if __name__ so it can be imported\nparser = argparse.ArgumentParser(description=\'Minimax Optimization.\')\nparser.add_argument(\'--training_time\',  type=float, default=10.0, help=\'total training time\')\nparser.add_argument(\'--n\', type=int, default=100, help=\'size of problem\')\nparser.add_argument(\'--m\', type=int, default=100, help=\'frequency of lazy Hessian update\')\nparser.add_argument(\'--seed\',  type=int, default=42, help=\'seed of random number\')\nparser.add_argument(\'--eta\', type=float, default=1e-2, help=\'stepsize of extra gradient\')\nparser.a

Average Metric: 23.00 / 143 (16.1%): : 154it [29:00, 118.74s/it]

In [ ]:
latest_program = best_progs[-1]

for name, pred in latest_program.named_predictors():
    print("================================")
    print(f"Predictor: {name}")
    print("================================")
    print("Prompt:")
    print(pred.signature.instructions)
    print("*********************************")

In [ ]:
# evaluate(latest_program)